# Raw to Bronze

Loads every CSV from the `adventurework.default.raw_data` volume into a Delta table in a new `adventurework.bronze` schema.

The raw CSVs are tab-separated and **headerless**, so the correct AdventureWorks column headers are extracted from the `CREATE TABLE` statements in `instawdb.sql` (also in the volume) and applied to each table.

**Run order: cell 1 -> cell 2 -> cell 3**

In [0]:
# Create the bronze schema
spark.sql("CREATE SCHEMA IF NOT EXISTS adventurework.bronze")
print("Schema adventurework.bronze is ready")

In [0]:
import re

RAW_DATA_PATH = "/Volumes/adventurework/default/raw_data"
SQL_SCRIPT_PATH = f"{RAW_DATA_PATH}/instawdb.sql"

# Maps each raw CSV file to its CREATE TABLE statement in instawdb.sql
CSV_TO_SQL_TABLE = {
    "Product (1).csv":         "Production.Product",
    "PurchaseOrderDetail.csv": "Purchasing.PurchaseOrderDetail",
    "PurchaseOrderHeader.csv": "Purchasing.PurchaseOrderHeader",
    "SalesOrderDetail.csv":    "Sales.SalesOrderDetail",
    "SalesTerritory.csv":      "Sales.SalesTerritory",
    "Vendor.csv":               "Purchasing.Vendor",
}

# Small configuration file -> driver-side read is fine
sql_script = open(SQL_SCRIPT_PATH, "r", encoding="utf-8-sig", errors="replace").read()

def extract_column_headers(schema_name, table_name):
    """Extract the ordered column headers of one CREATE TABLE statement from instawdb.sql."""
    match = re.search(re.escape(f"CREATE TABLE [{schema_name}].[{table_name}]("), sql_script)
    if not match:
        raise ValueError(f"CREATE TABLE [{schema_name}].[{table_name}] not found in instawdb.sql")
    # The column list ends at the first line that starts with ')'
    end = re.compile(r"^\)", re.M).search(sql_script, match.end())
    body = sql_script[match.end():end.start() if end else len(sql_script)]
    headers = []
    for line in body.split("\n"):
        column = re.match(r"\s*\[([^\]]+)\]", line)
        # Skip constraint definitions (PRIMARY KEY, FOREIGN KEY, CHECK, ...)
        if column and not re.match(r"\s*\[\s*(CONSTRAINT|PRIMARY|UNIQUE|CHECK|FOREIGN)\b", line, re.I):
            headers.append(column.group(1))
    return headers

csv_headers = {
    csv_file: extract_column_headers(*sql_table.split("."))
    for csv_file, sql_table in CSV_TO_SQL_TABLE.items()
}

for csv_file, headers in csv_headers.items():
    print(f"{csv_file}: {len(headers)} columns extracted")
    print(f"  {headers}")

In [0]:
import os

# Read each CSV (tab-separated, no header row) from the raw_data volume,
# apply the DDL column headers extracted above, and save as a bronze Delta table.
def bronze_table_name(csv_file):
    """'Product (1).csv' -> 'product', 'SalesOrderDetail.csv' -> 'salesorderdetail'"""
    stem = os.path.splitext(csv_file)[0]
    stem = re.sub(r"\s*\(\d+\)", "", stem)  # drop ' (1)' style suffixes
    return re.sub(r"[^a-z0-9]+", "_", stem.lower()).strip("_")

for csv_file, headers in csv_headers.items():
    table_name = bronze_table_name(csv_file)
    df = (spark.read
              .option("header", False)
              .option("sep", "\t")
              .option("inferSchema", True)
              .csv(f"{RAW_DATA_PATH}/{csv_file}"))
    assert len(df.columns) == len(headers), (
        f"{csv_file}: CSV has {len(df.columns)} columns but instawdb.sql defines {len(headers)}"
    )
    (df.toDF(*headers)
       .write
       .mode("overwrite")
       .saveAsTable(f"adventurework.bronze.{table_name}"))
    print(f"adventurework.bronze.{table_name}: {df.count()} rows, {len(headers)} columns")

print("\nBronze tables created:")
for t in sorted(spark.catalog.listTables("adventurework.bronze"), key=lambda t: t.name):
    print(f"  adventurework.bronze.{t.name}")